# Extracting the Time-Frequency Features

In [ ]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import json
import numpy as np
import pandas as pd
import shutil
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

from modules.datasets import ICBHIAudioDataset, KAUHAudioDataset
from modules.lungsound import LungSoundAudio
from modules.transforms import *

In [ ]:
DATA_PATH = Path(os.path.join(os.path.dirname(os.getcwd()), "data"))
INTERIM_DATA_FOLDER = DATA_PATH / "interim"
PREPROCESSED_DATA_FOLDER = DATA_PATH / "preprocessed"

if not os.path.exists(INTERIM_DATA_FOLDER):
    raise FileNotFoundError(f"Interim data folder not found at {INTERIM_DATA_FOLDER}. Please run the interim preprocessing step first.")

if not os.path.exists(PREPROCESSED_DATA_FOLDER):
    os.makedirs(PREPROCESSED_DATA_FOLDER)
    print(f"Created preprocessed data folder at {PREPROCESSED_DATA_FOLDER}.")
else:
    if len(os.listdir(PREPROCESSED_DATA_FOLDER)) > 0:
        print(f"[WARNING] Preprocessed data folder already exist and is not empty ({PREPROCESSED_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

## Preprocessed

In [ ]:
def save_features_as_npz(features: np.ndarray, sr: int, path: str):
    """
    Saves the extracted features and sampling rate to a .npz file.
    Args:
        features (np.ndarray): The extracted features to be saved.
        sr (int): The sampling rate associated with the features.
        path (str): The path where the features will be saved.
    """
    np.savez(path, features=features, sr=sr)


def preprocess_features(original_data_path: Path, preprocessed_data_path: Path, save_as: str = "npz"):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location using multiple feature extractors.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
        save_as (str): Format to save the preprocessed data ('npy', 'npz', or 'png').
    """
    datasets = [
        ICBHIAudioDataset(original_data_path),
        KAUHAudioDataset(original_data_path)
    ]

    feature_extractors = {
        "MagSTFT": MagSTFT(),
        "ImagSTFT": ImagSTFT(),
        "RealSTFT": RealSTFT(),
        "MelSpectrogram": MelSpectrogram(n_mels=128),
        "MFCC": MFCC(n_mfcc=128),
        "MFCCDelta": MFCCDelta(n_mfcc=128),
        "Chroma": Chroma(n_chroma=128),
        "Phase": Phase(),
    }

    # Iterate through each dataset and apply preprocessing with each feature extractor
    for dataset in datasets:
        df = dataset.data
        new_rows = []
        computed_data = False
        for feature_extractor_name, feature_extractor in feature_extractors.items():
            for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Preprocessing {dataset.name} with {feature_extractor.name}"):
                audio_path = Path(row["FilePath"])
                # Load the audio file using the LungSound class
                audio = LungSoundAudio(audio_path)
                # Extract features using the provided feature extractor
                features = feature_extractor(audio)

                # Construct the new file path for the preprocessed features
                diagnosis = str(row["Diagnosis"])
                new_file_name = f"{audio_path.stem}.{save_as}"
                preprocessed_file_path = preprocessed_data_path / dataset.name / feature_extractor_name / diagnosis / new_file_name
                preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)

                # Save the preprocessed audio to the new location
                if save_as == "npz":
                    save_features_as_npz(features.features, features.sr, preprocessed_file_path)
                else:
                    raise ValueError(f"Unsupported save format: {save_as}")

                # Add the row to the new df
                if not computed_data:
                    new_row = row.copy()
                    new_row["FilePath"] = new_file_name
                    new_rows.append(new_row)

            computed_data = True
            # Save the preprocessing parameters to a json file
            preprocessing_path = preprocessed_data_path / dataset.name / feature_extractor.name / "preprocessing.json"
            feature_extractor = {
                feature_extractor.__class__.__name__: {
                    "params": vars(feature_extractor),
                    "plot_params": feature_extractor.plot_params
                }
            }
            preprocessing = {
                "feature_extractor": feature_extractor,
            }
            with open(preprocessing_path, "w") as f:
                json.dump(preprocessing, f, indent=4, default=str)

        # Save the new data to a CSV file
        data_path = preprocessed_data_path / dataset.name / "data.csv"
        new_df = pd.DataFrame(new_rows).rename(columns={"FilePath": "FileName"})
        new_df.to_csv(data_path, index=False)

        print(f"Preprocessing for {dataset.name} completed. Total preprocessed audio files: {len(new_rows)}")
        print(f"Data saved to {os.path.relpath(data_path, start=os.getcwd())}")

In [ ]:
preprocess_features(INTERIM_DATA_FOLDER, PREPROCESSED_DATA_FOLDER, save_as="npz")